# 00 — Da pergunta à primeira resposta do Qwen

Este é o ponto de partida da trilha. Antes de adicionar documentos ou treinamento, será acompanhado o caminho de uma pergunta até a resposta: texto, template de conversa, tokens, tensores e geração autorregressiva. A saída deste notebook é uma execução registrada que servirá de referência para as etapas seguintes.

```mermaid
flowchart LR
 A[pergunta em português] --> B[template de mensagens]
 B --> C[tokenizer: texto para IDs]
 C --> D[Qwen 3B quantizado]
 D --> E[texto gerado]
```

A resposta ainda não consulta documentos. Isso dá uma linha de base: mais adiante, será possível observar o que muda quando evidências são acrescentadas (RAG) e quando o comportamento é ajustado (QLoRA).

## Preparar as dependências no kernel

Para que as bibliotecas sejam instaladas no mesmo Python que executa as células, usa-se o comando mágico `%pip` do Jupyter. A célula lê o `requirements.txt` do projeto e instala as versões declaradas. Depois da instalação, reinicie o kernel: bibliotecas recém-instaladas podem não ser carregadas corretamente por um processo Python que já estava aberto.

**O que observar:** o caminho de Python impresso precisa corresponder ao kernel escolhido. A instalação pode levar alguns minutos e requer conexão com o índice de pacotes.

---

### O que esta instalação altera

O comando instala bibliotecas no ambiente Python associado ao kernel. Por exemplo, se o kernel estiver usando Python A e o terminal estiver usando Python B, um pip comum no terminal pode instalar no B; o comando mágico do notebook instala no A. Por isso, o caminho de Python impresso na célula de diagnóstico é parte da reprodução do experimento.

In [3]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path

DIRETORIO_NOTEBOOK = Path.cwd()
if not (DIRETORIO_NOTEBOOK / "requirements.txt").is_file():
    DIRETORIO_NOTEBOOK = DIRETORIO_NOTEBOOK / "Agente_IA_Transformers_Credito"
CAMINHO_REQUIREMENTS = DIRETORIO_NOTEBOOK / "requirements.txt"
if not CAMINHO_REQUIREMENTS.is_file():
    raise FileNotFoundError(f"requirements.txt não encontrado em {CAMINHO_REQUIREMENTS}")

print("Instalando no kernel:", __import__("sys").executable)
print("Arquivo:", CAMINHO_REQUIREMENTS)
%pip install -r "{CAMINHO_REQUIREMENTS}"


Instalando no kernel: c:\Users\leova\anaconda3\envs\engenharia_ia_venv\python.exe
Arquivo: d:\Projetos_v2\Agente_IA_Transformers_Credito\requirements.txt
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached langchain_core-1.6.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached pypdf-6.16.2-py3-none-any.whl.metadata (7.5 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached streamlit-1.64.0-py3-none-any.whl.metadata (10 kB)
  Using cached uvicorn-0.34.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.7-py3-none-any.whl.metadata (22 kB)
  Using cached

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## 1. Por que começar com uma LLM local?

O projeto precisa de um gerador que explique Transformers em português e permita comparar respostas com e sem evidências documentais. Fixou-se `Qwen/Qwen2.5-3B-Instruct` para todas as condições. Essa decisão mantém o gerador constante: quando a qualidade mudar, a comparação poderá investigar o efeito do RAG ou do ajuste sem trocar simultaneamente de modelo.

Os pesos serão carregados em 4 bits para caber melhor na GPU de 8 GB. Isso reduz a memória dos pesos, mas não garante que toda a inferência ou o futuro treinamento caiba: cache de atenção, ativações e outras alocações também usam memória. A primeira resposta será uma demonstração técnica, não uma validação de qualidade no domínio de crédito.

---

### Quantização em termos concretos

Quantizar significa representar os pesos com menos bits. Para 3,09 bilhões de parâmetros, a conta teórica dos pesos é:

$$
\text{memória} \approx \text{parâmetros}\times\frac{\text{bits por parâmetro}}{8}
$$

Em 16 bits: aproximadamente 6,18 GB decimais. Em 4 bits: aproximadamente 1,55 GB decimais. Isso explica a escolha para a GPU de 8 GB, mas não inclui escalas da quantização, ativações nem cache de geração; a memória total será maior.

## 2. Conferir o Python e as bibliotecas

Antes de baixar pesos, confirma-se qual interpretador executa o notebook e se ele enxerga PyTorch, Transformers, Accelerate e bitsandbytes. O erro de pacote ausente costuma significar que o notebook está ligado a outro kernel ou que a instalação ainda não foi feita nesse ambiente.

O diagnóstico evita iniciar um download grande para depois descobrir que a configuração de quantização não pode ser criada. A célula seguinte também verifica CUDA e identifica a GPU disponível.

In [4]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python do kernel:", sys.executable)

pacotes = ["torch", "transformers", "accelerate", "bitsandbytes", "langchain-core"]
versoes = {}
for pacote in pacotes:
    try:
        versoes[pacote] = version(pacote)
        print(pacote, "->", versoes[pacote])
    except PackageNotFoundError as erro:
        raise RuntimeError(f"Falta {pacote} na venv engenharia_ia_venv. Instale as dependências nessa venv e reinicie o kernel.") from erro


Python do kernel: c:\Users\leova\anaconda3\envs\engenharia_ia_venv\python.exe
torch -> 2.6.0+cu126
transformers -> 5.8.1
accelerate -> 1.13.0
bitsandbytes -> 0.49.2
langchain-core -> 1.6.3


### Imports e dispositivo

Aqui são reunidas as ferramentas que serão usadas na inferência. PyTorch controla tensores e GPU; Transformers carrega o tokenizador, o modelo e a geração; bitsandbytes fornece a quantização de 4 bits. LangChain ainda não participa da geração: primeiro será mostrada a chamada direta para que as etapas não fiquem escondidas.

In [5]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Esta etapa requer PyTorch com CUDA no kernel selecionado.")

print("GPU:", torch.cuda.get_device_name(0))
memoria_livre, memoria_total = torch.cuda.mem_get_info(0)
print(f"VRAM total: {memoria_total / 1024**3:.2f} GiB")
print(f"VRAM livre agora: {memoria_livre / 1024**3:.2f} GiB")

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 2060 SUPER
VRAM total: 8.00 GiB
VRAM livre agora: 6.98 GiB


## 3. Fixar os parâmetros do experimento

O nome do modelo, a semente e os limites de entrada e saída ficam explícitos. O limite de tokens controla o custo e o tamanho do contexto desta primeira demonstração; não é o máximo teórico do modelo. A pasta de resultados é criada para guardar a execução que será comparada conceitualmente com as etapas futuras.

In [6]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
NOME_MODELO = "Qwen/Qwen2.5-3B-Instruct"
REVISAO_MODELO = "main"  # Na primeira execução; o commit resolvido será registrado.
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 256
set_seed(RANDOM_STATE)

# Aceita iniciar o notebook na própria pasta ou na raiz do repositório.
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 4. Como o texto vira tokens?

O modelo não recebe palavras diretamente. O tokenizador divide o texto em unidades do vocabulário e associa a cada unidade um número inteiro. Uma palavra pode virar um token ou vários; por isso, contagem de tokens não equivale a contagem de palavras.

A inspeção abaixo usa o mesmo checkpoint do gerador. A saída permite ver a relação texto → fragmentos → IDs antes de carregar os pesos da LLM.

---

### Exemplo: texto, tokens e IDs

Considere a frase “crédito PF”. O tokenizador pode dividi-la em fragmentos como [“cr”, “édito”, “ PF”] e converter cada fragmento em um ID inteiro. Essa divisão é ilustrativa: os fragmentos e IDs reais dependem do tokenizer do Qwen e são mostrados pela célula. O ponto importante é que o modelo recebe a sequência de IDs, não a frase como uma cadeia de caracteres.

In [7]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO, revision=REVISAO_MODELO)
if tokenizador.chat_template is None:
    raise ValueError("O checkpoint precisa fornecer o template de conversa.")

frase = "Transformers podem representar sequências de comportamento financeiro."
ids_frase = tokenizador.encode(frase, add_special_tokens=False)
tokens_frase = tokenizador.convert_ids_to_tokens(ids_frase)

print("Texto:", frase)
print("Quantidade de tokens:", len(ids_frase))
for token, identificador in zip(tokens_frase, ids_frase):
    print(repr(token), "->", identificador)
print("Texto reconstruído:", tokenizador.decode(ids_frase))

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\leova\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Texto: Transformers podem representar sequências de comportamento financeiro.
Quantidade de tokens: 13
'Transform' -> 8963
'ers' -> 388
'Ġpodem' -> 62676
'Ġrepresent' -> 4009
'ar' -> 277
'Ġsequ' -> 11256
'Ãªncias' -> 63809
'Ġde' -> 409
'Ġcomport' -> 51862
'amento' -> 10411
'Ġfinance' -> 17017
'iro' -> 8698
'.' -> 13
Texto reconstruído: Transformers podem representar sequências de comportamento financeiro.


**Como interpretar a saída:** os fragmentos exibidos podem incluir sinais que representam espaços ou partes de bytes. O caminho de volta é verificar a decodificação: se ela reconstrói a frase, os IDs representam a entrada esperada. O tokenizador só prepara texto para o modelo; ainda não calcula os embeddings usados para buscar documentos.

## 5. Montar as mensagens da conversa

Uma conversa Instruct separa papéis. A mensagem `system` define o comportamento desejado; a mensagem `user` contém a solicitação. Depois, o template do Qwen acrescenta marcadores próprios para indicar onde cada mensagem começa e termina e onde a geração deve continuar.

Essa instrução orienta a resposta durante a chamada. Ela não altera os pesos e não substitui treinamento nem evidências.

---

### Exemplo: papéis e template

Para a pergunta “O que é atenção?”, a conversa conceitual é:

| Papel | Conteúdo |
|---|---|
| system | Explique em português, com clareza e sem inventar fontes. |
| user | O que é atenção? |

O template do Qwen transforma essas mensagens em uma sequência com marcadores especiais de início, separação e fim. O marcador de geração deixa indicado que a próxima parte deve ser escrita pelo assistente.

In [8]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
INSTRUCAO_SISTEMA = (
    "Responda em português, de forma didática e impessoal. "
    "Explique Transformers no contexto de estudo de risco de crédito Pessoa Física. "
    "Diferencie fatos, hipóteses e exemplos sintéticos. "
    "Não invente artigos, métricas ou resultados. Quando não souber, explicite a limitação."
)
PERGUNTA = (
    "Explique brevemente por que a atenção pode ajudar a representar 12 meses "
    "de comportamento de um cliente PF. Dê uma hipótese de aplicação e "
    "um cuidado para evitar vazamento de informação."
)
mensagens = [
    {"role": "system", "content": INSTRUCAO_SISTEMA},
    {"role": "user", "content": PERGUNTA},
]
texto_prompt = tokenizador.apply_chat_template(
    mensagens, tokenize=False, add_generation_prompt=True,
)
print(texto_prompt)

<|im_start|>system
Responda em português, de forma didática e impessoal. Explique Transformers no contexto de estudo de risco de crédito Pessoa Física. Diferencie fatos, hipóteses e exemplos sintéticos. Não invente artigos, métricas ou resultados. Quando não souber, explicite a limitação.<|im_end|>
<|im_start|>user
Explique brevemente por que a atenção pode ajudar a representar 12 meses de comportamento de um cliente PF. Dê uma hipótese de aplicação e um cuidado para evitar vazamento de informação.<|im_end|>
<|im_start|>assistant



### Inspecionar o batch que será enviado

O template pronto é convertido em tensores. `input_ids` guarda os IDs dos tokens; `attention_mask` marca quais posições contêm entrada válida. Ambos têm forma `(B, T)`: B é o número de conversas no lote e T é o comprimento em tokens. Aqui há uma conversa, portanto B = 1.

A máscara externa do batch não deve ser confundida com os pesos de atenção internos das camadas Transformer.

---

### Exemplo de dimensões

Se uma conversa gera T = 37 tokens, então o lote terá input_ids com forma (1, 37): uma conversa e 37 posições. Se houver padding para combinar conversas de comprimentos diferentes, attention_mask marca quais posições são conteúdo e quais são preenchimento. A quantidade T real varia com o texto e o tokenizer.

In [9]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
entradas_cpu = tokenizador(
    texto_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)
quantidade_tokens_entrada = entradas_cpu["input_ids"].shape[1]
print("input_ids:", entradas_cpu["input_ids"].shape)
print("attention_mask:", entradas_cpu["attention_mask"].shape)
print("Primeiros IDs:", entradas_cpu["input_ids"][0, :20])
print("Tokens de entrada:", quantidade_tokens_entrada)
if quantidade_tokens_entrada > MAX_INPUT_TOKENS:
    raise ValueError("Prompt excede o orçamento inicial; reduza a pergunta antes de gerar.")

input_ids: torch.Size([1, 137])
attention_mask: torch.Size([1, 137])
Primeiros IDs: tensor([151644,   8948,    198,   1061,    618,   3235,    976,   2635,  29785,
         36830,     11,    409,  21791,   1521,  48502,    384,   3163,   9823,
           278,     13])
Tokens de entrada: 137


## 6. O que significa carregar em 4 bits?

Para N parâmetros armazenados com b bits cada, a memória teórica dos pesos é: M_pesos ≈ N × b/8 bytes.

Para 3,09 bilhões de parâmetros, 16 bits exigem cerca de 6,18 GB decimais; 4 bits exigem cerca de 1,55 GB. É apenas a memória dos pesos: metadados, ativações e cache de geração aumentam o consumo real.

NF4 representa os pesos de forma quantizada; as operações de cálculo usam FP16 nesta configuração. Inferência quantizada não é QLoRA: QLoRA será estudado quando adaptadores treináveis forem adicionados.

---

### NF4 e precisão de cálculo

NF4 é um formato de armazenamento quantizado que representa os pesos com quatro bits usando níveis escolhidos para distribuições de pesos de redes neurais. O cálculo das operações é feito em FP16 nesta configuração. Portanto, “pesos em 4 bits” não significa que toda a rede opera em 4 bits.

Exemplo: uma multiplicação pode ler pesos quantizados, reconstruir aproximações em blocos e acumular o resultado em uma precisão maior. Há economia no armazenamento dos pesos, com custo de conversão e metadados. QLoRA acrescentará adaptadores treináveis sobre essa base quantizada; nesta etapa há apenas inferênciaPara N parâmetros armazenados com b bits cada, a memória teórica dos pesos é M_pesos ≈ N × b/8 bytes.

Para 3,09 bilhões de parâmetros, 16 bits exigem cerca de 6,18 GB decimais; 4 bits exigem cerca de 1,55 GB. É apenas a memória dos pesos: metadados, ativações e cache de geração aumentam o consumo real.

---

### NF4 e precisão de cálculo

NF4 é um formato de armazenamento quantizado que representa os pesos com quatro bits usando níveis escolhidos para distribuições de pesos de redes neurais. O cálculo das operações é feito em FP16 nesta configuração. Portanto, “pesos em 4 bits” não significa que toda a rede opera em 4 bits.

Exemplo: uma multiplicação pode ler pesos quantizados, reconstruir aproximações em blocos e acumular o resultado em uma precisão maior. Há economia no armazenamento dos pesos, com custo de conversão e metadados. QLoRA acrescentará adaptadores treináveis sobre essa base quantizada; nesta etapa há apenas inferência.

In [10]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
configuracao_quantizacao = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

inicio_carregamento = time.perf_counter()
modelo = AutoModelForCausalLM.from_pretrained(
    NOME_MODELO,
    revision=REVISAO_MODELO,
    quantization_config=configuracao_quantizacao,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
modelo.eval()
tempo_carregamento = time.perf_counter() - inicio_carregamento
print(f"Carregamento: {tempo_carregamento:.1f} s")
print(f"Memória reportada para o modelo: {modelo.get_memory_footprint() / 1024**3:.2f} GiB")
print("Revisão resolvida:", getattr(modelo.config, "_commit_hash", None))

Loading weights: 100%|██████████| 434/434 [00:05<00:00, 75.13it/s] 


Carregamento: 157.9 s
Memória reportada para o modelo: 1.87 GiB
Revisão resolvida: aa8e72537993ba99e69dfaafa59ed015b17504d1


**Após o carregamento:** `eval()` coloca as camadas no modo de avaliação, enquanto `torch.inference_mode()` (usado na geração) desativa o acompanhamento de gradientes. São controles diferentes. O tempo de carregamento pode incluir download; nas execuções futuras, os arquivos em cache serão lidos do disco, mas o modelo ainda precisará ser enviado à memória da GPU.

## 7. Gerar sem consultar documentos

Os tensores da pergunta são enviados à GPU e o modelo prevê um token por vez. Em geração gulosa, escolhe-se em cada passo o token de maior probabilidade, sem amostragem. O resultado contém a entrada original seguida da continuação; por isso, a célula separa os novos tokens antes de decodificar a resposta.

Este é o baseline sem RAG. Se uma afirmação não tiver apoio, não há como rastreá-la a uma fonte ainda.

---

### Próximo token: exemplo pequeno

A cada passo, o modelo produz logits, isto é, pontuações não normalizadas para o próximo token. As probabilidades são obtidas com p = softmax(logits), e a geração gulosa escolhe argmax(p). Se três candidatos tiverem logits [1,2; 0,4; 2,1], a geração gulosa escolhe o terceiro, pois tem a maior pontuação. Esse token é acrescentado ao contexto e o processo se repete até o fim da resposta ou o limite configurado.

A escolha gulosa é reproduzível para a mesma entrada e configuração, mas não garante que a afirmação gerada seja verdadeira. Essa é uma das razões para acrescentar fontes no RAG.

A fatoração autorregressiva é P(x₁,…,xT) = ∏ₜ P(xₜ | x₁,…,xₜ₋₁): cada token é previsto a partir dos anteriores. A geração gulosa escolhe o token de maior probabilidade em cada passo, sem procurar a sequência globalmente melhor.

In [11]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
entradas_gpu = entradas_cpu.to("cuda:0")
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
inicio_geracao = time.perf_counter()

with torch.inference_mode():
    ids_saida = modelo.generate(
        **entradas_gpu,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizador.pad_token_id or tokenizador.eos_token_id,
    )

torch.cuda.synchronize()
tempo_geracao = time.perf_counter() - inicio_geracao
# A saída causal inclui o prompt; recorta-se apenas a continuação.
ids_resposta = ids_saida[0, quantidade_tokens_entrada:]
resposta = tokenizador.decode(ids_resposta, skip_special_tokens=True)
print(resposta)

Transformers são modelos de aprendizado de máquina capazes de processar sequências de dados, como textos ou fluxos de tempo. No contexto do risco de crédito de pessoa física, eles podem ser usados para modelar o comportamento financeiro de um cliente ao longo de vários anos. Isso é útil porque permite prever se um cliente tem histórico de pagamentos consistentes ou se há riscos de inadimplência.

**Fatos**: Um cliente com histórico de pagamento regular tende a ter menos riscos de inadimplência do que aquele com histórico irregular. 

**Hipótese**: Se um cliente tiver um histórico de pagamento consistente ao longo dos últimos 12 meses, isso aumenta as chances de ele continuar pagando seus empréstimos de maneira regular no futuro.

**Exemplo Sintético**: Imagine um cliente com 5 anos de histórico de pagamento. Se seu comportamento foi consistente nos últimos 12 meses (por exemplo, pagou todos os seus empréstimos no prazo), podemos usar um modelo Transformer para prever se ele continuará 

### Ler a resposta junto com o custo

A qualidade precisa ser inspecionada por critérios definidos: a explicação faz sentido? A hipótese de crédito está identificada como hipótese? O cuidado temporal evita leakage? O tempo e o pico de VRAM descrevem apenas esta chamada e este ambiente; uma execução isolada não é benchmark.

In [12]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
quantidade_tokens_saida = len(ids_resposta)
print("Tokens gerados:", quantidade_tokens_saida)
print(f"Tempo da geração: {tempo_geracao:.2f} s")
print(f"Tokens/s nesta chamada: {quantidade_tokens_saida / tempo_geracao:.2f}")
print(f"Pico alocado pelo PyTorch: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
if quantidade_tokens_saida == MAX_NEW_TOKENS:
    print("A resposta pode ter atingido o limite. Conferir se a explicação ficou incompleta.")

Tokens gerados: 256
Tempo da geração: 17.22 s
Tokens/s nesta chamada: 14.87
Pico alocado pelo PyTorch: 1.99 GiB
A resposta pode ter atingido o limite. Conferir se a explicação ficou incompleta.


## 8. Onde entra o LangChain?

Até agora, as mensagens foram montadas diretamente e a chamada ao Qwen permaneceu visível. O LangChain será usado para representar o mesmo par de mensagens em um template reutilizável. A comparação abaixo verifica se a forma organizada pelo framework resulta exatamente no mesmo texto de prompt.

Por enquanto, o LangChain organiza a entrada; não adiciona recuperação nem chama uma segunda LLM.

---

### Exemplo de template reutilizável

O template contém uma variável, por exemplo {pergunta}. Ao chamar o prompt com pergunta = “O que é atenção?”, essa variável é substituída pelo texto recebido e as mensagens system/user são formatadas. O teste de igualdade nesta célula verifica se o framework produziu o mesmo prompt que foi montado diretamente; assim, a abstração é conferida em vez de presumida.

In [13]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate

template_langchain = ChatPromptTemplate.from_messages([
    ("system", INSTRUCAO_SISTEMA),
    ("human", "{pergunta}"),
])
prompt_langchain = template_langchain.invoke({"pergunta": PERGUNTA})
mensagens_langchain = prompt_langchain.to_messages()

# Conversão explícita entre os nomes dos papéis do LangChain e do Qwen.
mensagens_qwen = [
    {"role": "system", "content": mensagens_langchain[0].content},
    {"role": "user", "content": mensagens_langchain[1].content},
]
texto_via_langchain = tokenizador.apply_chat_template(
    mensagens_qwen, tokenize=False, add_generation_prompt=True,
)
print(texto_via_langchain)
assert texto_via_langchain == texto_prompt
print("O template organizado pelo LangChain preservou o prompt da chamada direta.")

<|im_start|>system
Responda em português, de forma didática e impessoal. Explique Transformers no contexto de estudo de risco de crédito Pessoa Física. Diferencie fatos, hipóteses e exemplos sintéticos. Não invente artigos, métricas ou resultados. Quando não souber, explicite a limitação.<|im_end|>
<|im_start|>user
Explique brevemente por que a atenção pode ajudar a representar 12 meses de comportamento de um cliente PF. Dê uma hipótese de aplicação e um cuidado para evitar vazamento de informação.<|im_end|>
<|im_start|>assistant

O template organizado pelo LangChain preservou o prompt da chamada direta.


## 9. Registrar para reproduzir

A execução será salva em JSON com modelo, revisão, versões, prompt, resposta e custos observados. Esse registro permite saber exatamente qual configuração produziu a saída. Ele não atribui uma nota de qualidade nem transforma uma demonstração em evidência de desempenho.

In [14]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
registro = {
    "modelo": NOME_MODELO,
    "revisao": getattr(modelo.config, "_commit_hash", None),
    "seed": RANDOM_STATE,
    "versoes": versoes,
    "gpu": torch.cuda.get_device_name(0),
    "quantizacao": "NF4, double quant, calculo FP16",
    "max_new_tokens": MAX_NEW_TOKENS,
    "prompt": texto_prompt,
    "resposta": resposta,
    "tokens_entrada": quantidade_tokens_entrada,
    "tokens_saida": quantidade_tokens_saida,
    "tempo_geracao_s": tempo_geracao,
    "pico_pytorch_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "usa_rag": False,
    "usa_adaptador": False,
}
caminho_resultado = DIRETORIO_RESULTADOS / "primeira_resposta_qwen.json"
caminho_resultado.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print("Registro salvo:", caminho_resultado)

Registro salvo: d:\Projetos_v2\Agente_IA_Transformers_Credito\resultados\primeira_resposta_qwen.json


## 10. Fechamento: o que já foi construído e o que falta

Até aqui, uma pergunta virou tokens, foi organizada no formato de conversa e recebeu uma resposta do Qwen. Ainda falta fornecer conhecimento verificável. A etapa 01 começa esse próximo elo: PDFs → texto extraído → páginas rastreáveis → chunks → embeddings. A etapa 02 persistirá esses vetores no Chroma.

| Já observado | Próxima pergunta do projeto |
|---|---|
| Como o Qwen recebe uma pergunta | Como localizar trechos úteis nos documentos? |
| Como funciona a geração sem fontes | O que muda ao fornecer evidências? |
| Como medir tokens e memória de uma chamada | O RAG ou o ajuste melhora respostas avaliadas? |

Ajustar uma pergunta pode ser útil para explorar o modelo carregado, mas o comparativo formal virá depois, com um conjunto reservado.

---

### Ligação com a próxima etapa

A próxima etapa não troca o Qwen. Ela constrói uma fonte externa consultável. O mesmo conceito pode então ser perguntado sem documentos e com trechos recuperados, permitindo estudar o papel da evidência separadamente do papel dos pesos do gerador.